In [3]:
import logging
logging.getLogger('pgmpy').setLevel(logging.ERROR)

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import pandas as pd

# Sample dataset
data = pd.DataFrame({
    'Income': ['High', 'High', 'Low', 'Medium', 'Low', 'Medium', 'High', 'Low'],
    'CreditScore': ['Good', 'Good', 'Bad', 'Good', 'Bad', 'Bad', 'Good', 'Bad'],
    'Employment': ['Stable', 'Unstable', 'Stable', 'Stable', 'Unstable', 'Stable', 'Stable', 'Unstable'],
    'LoanApproved': ['Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No']
})

model = DiscreteBayesianNetwork([
    ('Income', 'LoanApproved'),
    ('CreditScore', 'LoanApproved'),
    ('Employment', 'LoanApproved')
])

model.fit(data, estimator=MaximumLikelihoodEstimator)

assert model.check_model()

print("=== CPDs ===")
for cpd in model.get_cpds():
    print(cpd)

inference = VariableElimination(model)

result = inference.query(
    variables=['LoanApproved'],
    evidence={
        'Income': 'Low',
        'CreditScore': 'Bad',
        'Employment': 'Unstable'
    }
)

print("\n=== Loan Approval Prediction ===")
print(result)

=== CPDs ===
+----------------+-------+
| Income(High)   | 0.375 |
+----------------+-------+
| Income(Low)    | 0.375 |
+----------------+-------+
| Income(Medium) | 0.25  |
+----------------+-------+
+-------------------+-----+----------------------+
| CreditScore       | ... | CreditScore(Good)    |
+-------------------+-----+----------------------+
| Employment        | ... | Employment(Unstable) |
+-------------------+-----+----------------------+
| Income            | ... | Income(Medium)       |
+-------------------+-----+----------------------+
| LoanApproved(No)  | ... | 0.5                  |
+-------------------+-----+----------------------+
| LoanApproved(Yes) | ... | 0.5                  |
+-------------------+-----+----------------------+
+-------------------+-----+
| CreditScore(Bad)  | 0.5 |
+-------------------+-----+
| CreditScore(Good) | 0.5 |
+-------------------+-----+
+----------------------+-------+
| Employment(Stable)   | 0.625 |
+----------------------+-------